<a href="https://colab.research.google.com/github/Anoushehm/intro-ml-course-winter2026/blob/main/4-Datathon3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HAD5016 - Datathon #3 - High-fidelity
**Project:** This colab file aims to predict the mortality in the first 24 hours for the visitors of the intensive care unit (ICU) using the vital, lab, and Apache covariates. To do so, we will use the dataset that includes 91,000 intensive care unit (ICU) visits that are collected from various hospitals, covering an entire year. These hospitals are located in Argentina, Australia, New Zealand, Sri Lanka, Brazil, and over 200 hospitals in the United States. This dataset also contains apache IV scores which is a highly accurate predictor of mortality. Our final goal is to use the features available in this dataset to predict mortality and compare the prediction performance of our model to the apache score available in the dataset.



**Programmer:** Chieh En Chen, Neelan Sriranjan, Anousheh Marouzi

**Team Number:** 4


**Date started:** March 1st, 2026

**Last update:** March 4th, 2026

In [ ]:
from google.colab import files
import io
import pandas as pd

uploaded = files.upload()

filename = next(iter(uploaded))
print(f"Reading: {filename}")

df = pd.read_csv(io.BytesIO(uploaded[filename]))

df.head()

In [ ]:
# importing PyTorch library as 't' for convenience
import torch as t

# importing other necessary libraries
import numpy as np
from torch.nn.functional import sigmoid, relu, tanh
from torch.optim import Adam
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

# importing PyTorch modules for building neural networks
from torch.nn import Tanh, Linear, Sequential, Sigmoid, Dropout

In [ ]:
import pandas as pd
import numpy as np

X_cols = [
    'age',
    'elective_surgery',
    'gender',
    'pre_icu_los_days',
    'readmission_status',
    'albumin_apache',
    'apache_post_operative',
    'arf_apache',
    'bilirubin_apache',
    'bun_apache',
    'gcs_eyes_apache',
    'gcs_motor_apache',
    'gcs_verbal_apache',
    'glucose_apache',
    'hematocrit_apache',
    'intubated_apache',
    'sodium_apache',
    'urineoutput_apache',
    'wbc_apache',
    'd1_heartrate_max',
    'd1_mbp_min',
    'd1_resprate_max',
    'd1_spo2_min',
    'd1_temp_max',
    'd1_platelets_min',
    'immunosuppression',
    'solid_tumor_with_metastasis'
]

Y_cols = ['hospital_death']

X = df[X_cols].values
Y = df[Y_cols].values
print("X shape:", X.shape)
print("Y shape:", Y.shape)

df[X_cols + Y_cols].dtypes


**Data Exploration**

In [ ]:
print("\n[Descriptive Statistics]")
display(df[X_cols].describe())
display(df[Y_cols].describe())

In [ ]:
# ---------------------------------------------------------------------------- #
# Check for duplicates and unique ids
# ---------------------------------------------------------------------------- #

# Count the number of observations
num_observations = len(df)
print(f"Number of observations: {num_observations}")

# Count the number of duplicated observations
num_duplicates = df.duplicated().sum()
print(f"Number of duplicated observations: {num_duplicates}")

# Calculate the percentage of duplicated rows
percentage_duplicates = (num_duplicates / num_observations) * 100
print(f"Percentage of duplicated rows: {percentage_duplicates:.2f}%")

# Get unique number of patient id
unique_patient_id = df['patient_id'].nunique()
print(f"Number of unique patient id: {unique_patient_id}")

# Get unique number of hospital id
unique_hospital_id = df['hospital_id'].nunique()
print(f"Number of unique hospital id: {unique_hospital_id}")

# Get a unique number of icu_id
unique_icu_id = df['icu_id'].nunique()
print(f"Number of unique icu_id: {unique_icu_id}")

# Calculate number of duplicated icu_id
num_duplicates_icu_id = df.duplicated(subset=['icu_id']).sum()
print(f"Number of duplicated icu_id: {num_duplicates_icu_id}")

In [ ]:
# ---------------------------------------------------------------------------- #
# Check for missing values
# ---------------------------------------------------------------------------- #

# Check that all selected variables exist
X_cols_original = [
    'age', 'elective_surgery', 'gender', 'pre_icu_los_days', 'readmission_status',
    'albumin_apache', 'apache_2_diagnosis', 'apache_3j_diagnosis', 'apache_post_operative',
    'arf_apache', 'bilirubin_apache', 'bun_apache', 'creatinine_apache', 'fio2_apache',
    'gcs_eyes_apache', 'gcs_motor_apache', 'gcs_unable_apache', 'gcs_verbal_apache',
    'glucose_apache', 'heart_rate_apache', 'hematocrit_apache', 'intubated_apache',
    'map_apache', 'paco2_apache', 'paco2_for_ph_apache', 'pao2_apache', 'ph_apache',
    'resprate_apache', 'sodium_apache', 'temp_apache', 'urineoutput_apache',
    'ventilated_apache', 'wbc_apache', 'd1_heartrate_max', 'd1_mbp_min',
    'd1_resprate_max', 'd1_spo2_min', 'd1_temp_max', 'd1_lactate_max',
    'd1_platelets_min', 'immunosuppression', 'solid_tumor_with_metastasis'
]
missing_columns = set(X_cols_original) - set(df.columns)

if len(missing_columns) > 0:
    raise ValueError(f"The following variables are missing from the dataset: {missing_columns}")

print("All selected variables exist in the dataset.\n")

# Calculate missing values
missing_summary = pd.DataFrame({
    "Missing_Count": df[X_cols_original].isna().sum(),
    "Missing_Percent": df[X_cols_original].isna().mean() * 100
})

# Sort by highest missing percentage
missing_summary = missing_summary.sort_values(by="Missing_Percent", ascending=False)

print("Missing values summary:")
print(missing_summary)

**Data Cleaning-cleaned dataframe: df2**

1.   Drop rows where 'pre_icu_los_days' is less than 0
2.   Create a new binary variable 'd1_temp_max_high': 1 if 'd1_temp_max' > 38, else 0
3.  Create a new binary variable 'd1_spo2_min_low': 1 if 'd1_spo2_min' < 90, else 0



In [ ]:
# Drop rows where 'pre_icu_los_days' is less than 0
df2 = df[df['pre_icu_los_days'] >= 0].copy()

# Create a new binary variable 'd1_temp_max_high': 1 if 'd1_temp_max' > 38, else 0
df2['d1_temp_max_high'] = (df2['d1_temp_max'] > 38).astype(int)

# Create a new binary variable 'd1_spo2_min_low': 1 if 'd1_spo2_min' < 90, else 0
df2['d1_spo2_min_low'] = (df2['d1_spo2_min'] < 90).astype(int)

print("Shape of DataFrame after dropping rows:", df2.shape)
print("\nFirst 5 rows with new binary variables:")
display(df2[['pre_icu_los_days', 'd1_temp_max', 'd1_temp_max_high', 'd1_spo2_min', 'd1_spo2_min_low']].head())

print("\nValue counts for 'd1_temp_max_high':")
print(df2['d1_temp_max_high'].value_counts())

print("\nValue counts for 'd1_spo2_min_low':")
print(df2['d1_spo2_min_low'].value_counts())

In [ ]:
# ---------------------------------------------------------------------------- #
# Describe clean and ready-to-analysis data frame
# ---------------------------------------------------------------------------- #

# Count the number of observations
num_observations = len(df2)
print(f"Number of observations: {num_observations}")

# Calculate the average age
average_age = df2['age'].mean()
print("\n[Average Age]")
print(average_age)

# Calculate the percentage for gender
gender_counts = df2['gender'].value_counts()
gender_percentages = gender_counts / len(df2)
print("\n[Gender Distribution]")
print(gender_percentages)

# Calcuate the number of missing values for ethnicity
num_missing_ethnicity = df2['ethnicity'].isna().sum()
print("\n[Number of Missing Values for Ethnicity]")
print(num_missing_ethnicity)

# Calculate the percentage of missing values for ethnicity
percentage_missing_ethnicity = (num_missing_ethnicity / num_observations) * 1
print("\n[Percentage of Missing Values for Ethnicity]")
print(percentage_missing_ethnicity)

# Calculate the percentage for ethnicity categories
ethnicity_counts = df2['ethnicity'].value_counts()
ethnicity_percentages = ethnicity_counts / len(df2)
print("\n[Ethnicity Distribution]")
print(ethnicity_percentages)



**[Visualization]Data Exploration after data cleaning **

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Use the cleaned DataFrame
df2_vis = df2

# Define the target column
target_col = 'hospital_death'

# Select numerical columns for visualization. Excluding 'gender' as it's an object type.
# Choosing a subset to fit the plot layout.
numerical_columns = [
    'age', 'pre_icu_los_days', 'wbc_apache', 'heart_rate_apache',
    'd1_temp_max', 'd1_spo2_min'
]

# Boxplots
print("\n[Visualizing Outliers - Boxplots]")
# Ensure only existing columns are plotted
columns_to_plot_boxplot = [col for col in numerical_columns if col in df2_vis.columns]
if not columns_to_plot_boxplot:
    print("No numerical columns found to plot boxplots.")
else:
    df2_vis[columns_to_plot_boxplot].plot(kind='box', subplots=True, layout=(2, 3), figsize=(15, 8), color='#7569c9')
    plt.tight_layout()
    plt.show()

# Histograms
print("\n[Visualizing Distributions - Histograms]")
fig, axs = plt.subplots(ncols=3, nrows=2, figsize=(15, 10))
# Flatten axis array for easy iteration
axs = axs.flatten()

# Ensure only existing columns are plotted
columns_to_plot_hist = [col for col in numerical_columns if col in df2_vis.columns]

if not columns_to_plot_hist:
    print("No numerical columns found to plot histograms.")
else:
    for i, column in enumerate(columns_to_plot_hist):
        if i < len(axs):
            sns.histplot(data=df2_vis, x=column, hue=target_col, kde=True, palette='rocket', ax=axs[i], common_norm=False)
            axs[i].set_title(f'Distribution of {column}')

    # Turn off unused subplots
    for i in range(len(columns_to_plot_hist), len(axs)):
        if i < len(axs):
            axs[i].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Calcualte the percentage for gender
gender_counts = df2['gender'].value_counts()
gender_percentages = gender_counts / len(df2)
print("\n[Gender Distribution]")
print(gender_percentages)


# Calculate the average age
average_age = df2['age'].mean()
print("\n[Average Age]")
print(average_age)

# Calculate the percentage of for ethnicity
ethnicity_counts = df2['ethnicity'].value_counts()
ethnicity_percentages = ethnicity_counts / len(df2)
print("\n[Ethnicity Distribution]")
print(ethnicity_percentages)




**Class Imbalance**

In [ ]:
### Create a pie chart to show the distribution of categories in the 'hospital_death' variable
counts = df2[target_col].value_counts()

# Define colors
colors = ['#1c3a73', '#7cb1c2']

# Create pie chart
plt.figure(figsize=(6, 6))
plt.pie(
    counts, #counts for each class
    labels=[f'{cls} ({count})' for cls, count in counts.items()],  # add labels with counts... counts.items() returns an iterable of tuples
                                                                    #e.g., [('class', count)], which are then unpacked and formatted into labels
    autopct='%1.1f%%',  # Show percentages
    colors=colors,
    startangle=90
)

# Set title
plt.title('Distribution of Classes in "hospital_death"', fontsize=14)

# Show plot
plt.show()

**Data Preparation**

In [ ]:
# Randomly sample 70% of the data for training
train_df2 = df2.sample(frac = .7, random_state=10)
# Use the remaining 30% for testing
test_df2 = df2.drop(train_df2.index)

# Check the mean of the 'hospital_death' column in both training and testing data
print(train_df2['hospital_death'].mean())
print(test_df2['hospital_death'].mean())

# Data Preparation

# Define the full list of feature columns for the model
# First, create a temporary list from X_cols, excluding the original 'd1_temp_max' and 'd1_spo2_min'
X_cols_cleaned = [col for col in X_cols if col not in ['d1_temp_max', 'd1_spo2_min']]
# Then, add the new binary features to this cleaned list
final_feature_cols = X_cols_cleaned + ['d1_temp_max_high', 'd1_spo2_min_low']

# Extract the 'hospital_death' column as the target variable for training and testing
Y_train = train_df2['hospital_death'].to_numpy()
Y_test = test_df2['hospital_death'].to_numpy()

# Create feature DataFrames for training and testing, selecting only the desired columns
X_train_df = train_df2[final_feature_cols]
X_test_df = test_df2[final_feature_cols]

# Identify categorical columns within these feature sets.
categorical_cols_to_encode = X_train_df.select_dtypes(include=['object', 'category']).columns.tolist()

if categorical_cols_to_encode:
    print(f"One-hot encoding categorical columns: {categorical_cols_to_encode}")
    # Apply one-hot encoding
    X_train_df = pd.get_dummies(X_train_df, columns=categorical_cols_to_encode, drop_first=True, dtype=int)
    X_test_df = pd.get_dummies(X_test_df, columns=categorical_cols_to_encode, drop_first=True, dtype=int)

    # Align columns after one-hot encoding to ensure both train and test sets have the same columns
    train_cols_after_ohe = set(X_train_df.columns)
    test_cols_after_ohe = set(X_test_df.columns)

    missing_in_test = list(train_cols_after_ohe - test_cols_after_ohe)
    for c in missing_in_test:
        X_test_df[c] = 0

    missing_in_train = list(test_cols_after_ohe - train_cols_after_ohe)
    for c in missing_in_train:
        X_train_df[c] = 0

    # Ensure the order of columns is the same for both datasets
    X_test_df = X_test_df[X_train_df.columns]
else:
    print("No categorical columns found for one-hot encoding.")

# Impute missing values with the mean after one-hot encoding but before scaling
# Calculate mean from training data and apply to both train and test
for col in X_train_df.columns:
    if X_train_df[col].isnull().any():
        mean_val = X_train_df[col].mean()
        X_train_df[col] = X_train_df[col].fillna(mean_val)
        X_test_df[col] = X_test_df[col].fillna(mean_val)


# Convert the feature DataFrames to numpy arrays
X_train = X_train_df.to_numpy()
X_test = X_test_df.to_numpy()

# Standardize the features to have zero mean and unit variance
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

print("X_train shape after processing:", X_train.shape)
print("X_test shape after processing:", X_test.shape)

try:
    import miceforest as mf
except ImportError:
    !pip install miceforest
    import miceforest as mf


def apply_imputation_instructions(
    df: pd.DataFrame,
    mice_cols=None,
    n_imputations: int = 5,
    n_iterations: int = 5,
    random_state: int = 42,
) -> pd.DataFrame:
    """ Imputation rules:
          - Mean impute: Age
          - Fixed-value imputes for binary/GCS/etc
          - Derive binary flags from vitals/labs thresholds; missing -> 0
          - MICE imputation for specified continuous vars
        Returns a NEW dataframe (does not modify input).
    """

    out = df.copy()

    # ----------------------------
    # 1) Simple imputations / fixed defaults
    # ----------------------------

    # Age: missing -> average
    if "Age" in out.columns:
        out["Age"] = pd.to_numeric(out["Age"], errors="coerce")
        out["Age"] = out["Age"].fillna(out["Age"].mean())

    # Binary vars with fixed missing defaults
    fill_zero_cols = [
        "Elective_surgery",               # missing -> 0
        "Arf_apache",                     # missing -> 0
        "Intubated_apache",               # missing -> 0
        "Immunosuppression",              # missing -> 0
        "Solid_tumor_with_metastasis",    # missing -> 0
    ]
    for c in fill_zero_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int)

    # Gender: 1/0, missing -> 1
    if "Gender" in out.columns:
        out["Gender"] = pd.to_numeric(out["Gender"], errors="coerce").fillna(1).astype(int)

    # GCS components: fixed defaults
    gcs_defaults = {
        "Gcs_eyes_apache": 4,
        "GCS_verbal_apache": 5,
        "GCS_motor_apache": 6,
    }
    for c, v in gcs_defaults.items():
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(v)

    # ----------------------------
    # 2) Threshold-derived binary flags (missing -> 0)
    # ----------------------------
    def make_flag(series, condition, missing_value=0):
        s = pd.to_numeric(series, errors="coerce")
        flag = np.where(s.isna(), missing_value, condition(s).astype(int))
        return pd.Series(flag, index=series.index)

    # D1_heartrate_max -> >100 is 1, else 0, missing -> 0
    if "D1_heartrate_max" in out.columns:
        out["D1_heartrate_max_flag"] = make_flag(out["D1_heartrate_max"], lambda s: s > 100, 0)

    # D1_mbp_min -> <65 is 1, else 0, missing -> 0
    if "D1_mbp_min" in out.columns:
        out["D1_mbp_min_flag"] = make_flag(out["D1_mbp_min"], lambda s: s < 65, 0)

    # D1_resprate_max -> >20 is 1, else 0, missing -> 0
    if "D1_resprate_max" in out.columns:
        out["D1_resprate_max_flag"] = make_flag(out["D1_resprate_max"], lambda s: s > 20, 0)

    # D1_spo2_min -> <90 is 1, else 0, missing -> 0
    if "D1_spo2_min" in out.columns:
        out["D1_spo2_min_flag"] = make_flag(out["D1_spo2_min"], lambda s: s < 90, 0)

    # D1_temp_max -> >38 is 1, else 0, missing -> 0
    if "D1_temp_max" in out.columns:
        out["D1_temp_max_flag"] = make_flag(out["D1_temp_max"], lambda s: s > 38, 0)

    # D1_platelets_min -> <150 is 1 and >=150 is 0, missing -> 0
    if "D1_platelets_min" in out.columns:
        out["D1_platelets_min_flag"] = make_flag(out["D1_platelets_min"], lambda s: s < 150, 0)



    # ----------------------------
    # 3) MICE for selected continuous variables
    # ----------------------------
    # As specified:
    # Pre-icu_los_days, Bilirubin_apache, Bun_apache, Glucose_apache, Hematocrit_apache,
    # Sodium_apache, Urineoutput_apache, Wbc_apache
    if mice_cols is None:
        mice_cols = [
            "Pre-icu_los_days",
            "Bilirubin_apache",
            "Bun_apache",
            "Glucose_apache",
            "Hematocrit_apache",
            "Sodium_apache",
            "Urineoutput_apache",
            "Wbc_apache",
        ]

    # Keep only those that exist in df
    mice_cols = [c for c in mice_cols if c in out.columns]

    if len(mice_cols) > 0:
        # miceforest expects numeric dtypes for these columns
        for c in mice_cols:
            out[c] = pd.to_numeric(out[c], errors="coerce")

        # Build kernel on ONLY the MICE columns (simple + robust).
        # If you want MICE to use other predictors too, include them here.
        mice_data = out[mice_cols].copy()

        # Only run MICE if there's actually missingness
        if mice_data.isna().any().any():
            kernel = mf.ImputationKernel(
                mice_data,
                datasets=n_imputations,
                save_all_iterations=True,
                random_state=random_state,
            )
            kernel.mice(iterations=n_iterations)

            # Use the first completed dataset (index 0)
            completed = kernel.complete_data(dataset=0)

            # Put back into output
            out[mice_cols] = completed[mice_cols]

    # Final tidy: ensure binary columns are ints where appropriate
    binary_like = [
        "Elective_surgery",
        "Gender",
        "Arf_apache",
        "Intubated_apache",
        "Immunosuppression",
        "Solid_tumor_with_metastasis",
        # flags
        "D1_heartrate_max_flag",
        "D1_mbp_min_flag",
        "D1_resprate_max_flag",
        "D1_spo2_min_flag",
        "D1_temp_max_flag",
        "D1_platelets_min_flag",
    ]
    for c in binary_like:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int)

    return out

**Building a Neural Network in PyTorch**

In [ ]:
# Define the number of neurons(=the number of features) in the first and second hidden layers
hidden_units_layer_1 = 27
hidden_units_layer_2 = 27

# FIRST LAYER: Define weights and biases for the first layer
W1 = t.randn((X_train.shape[1], hidden_units_layer_1), requires_grad=True)
B1 = t.zeros((1, hidden_units_layer_1), requires_grad=True) #each neuron gets 1 bias term

# SECOND LAYER: Define weights and biases for the second layer
W2 = t.randn((hidden_units_layer_1, hidden_units_layer_2), requires_grad=True)
B2 = t.zeros((1, hidden_units_layer_2), requires_grad=True)

# THIRD LAYER: Define weights and biases for the output layer
W3 = t.randn((hidden_units_layer_2, 1), requires_grad=True)
B3 = t.zeros((1, 1), requires_grad=True)

Defining the Forward Pass of a Neural Network in PyTorch

In [ ]:
# Define the forward pass of the neural network
def forward(input):
    # First hidden layer with tanh activation
    out = tanh(input @ W1 + B1)

    # Second hidden layer with tanh activation
    out = tanh(out @ W2 + B2)

    # Output layer with sigmoid activation (since it's a binary classification problem)
    out = sigmoid(out @ W3 + B3)
    return out

Preparing Data and Training Utilities for Neural Network Training in PyTorch

In [ ]:
# Convert the training data to PyTorch tensors
X = t.Tensor(X_train).type(t.float32)
Y = t.Tensor(Y_train).type(t.float32)

# Create a dataset from tensors to be used with DataLoader
train_dataset = TensorDataset(X, Y)

# Define training hyperparameters
epochs = 500 # Reduced epochs for faster execution
learning_rate = 0.001 # Reduced learning rate to prevent instability
batch_size = 32

# DataLoader provides batches of data for training
train_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Define the optimizer (Adam) and include all weights and biases
optimizer = Adam([W1, B1, W2, B2, W3, B3], lr=learning_rate)

# Define the loss function (Binary Cross-Entropy Loss)
loss_fn = t.nn.BCELoss()

Training Loop for a Neural Network in PyTorch

In [ ]:
train_loss_list = []

# Train the model for a specified number of epochs
for epoch in range(epochs):
    # Reduce the learning rate every 500 epochs
    if epoch % 500 == 0:
        learning_rate *= .9

    per_epoch_loss_list = []

    # Iterate over all batches of data
    for batch_idx, (X, Y) in enumerate(train_data_loader):
        # Forward pass: Compute predictions
        probs = forward(X)

        # Compute the loss
        loss = loss_fn(probs.view(-1), Y)

        # Backward pass: Compute gradient and update weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Record the loss for this batch
        per_epoch_loss_list.append(loss.item())

    # Record the average loss for this epoch
    train_loss_list.append(sum(per_epoch_loss_list) / len(per_epoch_loss_list))

In [ ]:
# Plot the training loss over epochs
plt.plot([i for i in range(len(train_loss_list))], train_loss_list)
plt.xlabel('epochs')
plt.ylabel('loss')

**Evaluating Model Performance on Validation Data**

In [ ]:
# Evaluate Model Performance on Validation Data

# Disable gradient calculations for evaluation using t.no_grad()
with t.no_grad():
    # Prepare the validation data
    X = t.Tensor(X_test).type(t.float32)  # Convert validation features to a PyTorch tensor
    Y = t.Tensor(Y_test).type(t.float32)  # Convert validation labels to a PyTorch tensor

    # Calculate predictions on the validation data
    probs = forward(X)  # Pass validation data through the trained model
    loss = loss_fn(probs.view(-1), Y)  # Compute the loss between predictions and actual labels

    # Print the validation loss
    print('Validation Loss (fyi - lower value denotes better performance):')
    print(loss.item())

    # Now, evaluate the model on the training data
    X = t.Tensor(X_train).type(t.float32)  # Convert training features to a PyTorch tensor
    Y = t.Tensor(Y_train).type(t.float32)  # Convert training labels to a PyTorch tensor

    # Calculate predictions on the training data
    probs = forward(X)  # Pass training data through the trained model
    loss = loss_fn(probs.view(-1), Y)  # Compute the loss between predictions and actual labels

    # Print the training loss
    print('vs. Training Loss:')
    print(loss.item())

**Regularized Training of a Neural Network in PyTorch**

In [ ]:
import torch.nn as nn

# Regularization
# Deep neural networks are prone to overfitting if there is not much data available during training.

# Define hyperparameters
number_of_input_features = 27
number_of_hidden_units = 27
epochs = 100
learning_rate = 0.01
batch_size = 32
landa = 0.01  # Regularization term (lambda)

# Prepare the data
X_train_tensor = t.Tensor(X_train).type(t.float32)
Y_train_tensor = t.Tensor(Y_train).type(t.float32)

X_test_tensor = t.Tensor(X_test).type(t.float32)
Y_test_tensor = t.Tensor(Y_test).type(t.float32)

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
train_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Define the neural network model with regularization
model = nn.Sequential(
    nn.Linear(number_of_input_features, number_of_hidden_units),  # Linear layer for matrix multiplication and bias addition
    nn.Tanh(),  # Tanh activation function
    nn.Linear(number_of_hidden_units, 1),  # Another linear layer
    nn.Sigmoid()  # Sigmoid activation for probability output
)

# Define the optimizer
optimizer = Adam(model.parameters(), lr=learning_rate)

# Define the loss function with Binary Cross-Entropy Loss
loss_fn = nn.BCELoss()

# Lists to store training accuracy, validation accuracy, and training loss over epochs
train_accuracy_list = []
validation_accuracy_list = []
train_loss_list = []

for epoch in range(epochs):
    if epoch % 500 == 0:
        learning_rate *= 0.9  # Learning rate scheduling

    per_epoch_loss_list = []

    for batch_idx, (X, Y) in enumerate(train_data_loader):
        # Forward pass: Compute predictions
        probs = model(X)

        # Adding regularization term for all parameters in the model
        l2_term = sum([(w ** 2).sum() for w in model.parameters()])

        # New loss is the old loss + regularization term
        loss = loss_fn(probs.view(-1), Y) + landa * l2_term

        per_epoch_loss_list.append(loss.item())

        # Backward pass: Compute gradient and update weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate the model on training and validation data
    with t.no_grad():
        # Set the model in eval mode; some layers use this for certain calculations during training
        model.eval()

        # Calculate accuracy on train data
        probs = model(X_train_tensor)
        prediction = (probs >= 0.5).type(t.LongTensor).view(-1)
        train_accuracy = (prediction == Y_train_tensor).type(t.float32).mean().item()

        # Calculate accuracy on validation data
        probs = model(X_test_tensor)
        prediction = (probs >= 0.5).type(t.LongTensor).view(-1)
        validation_accuracy = (prediction == Y_test_tensor).type(t.float32).mean().item()

        # Print accuracy for the current epoch
        print(f'Epoch {epoch}/{epochs} ---> Train Accuracy: {train_accuracy}, Validation Accuracy: {validation_accuracy}')

        # Set the model back to train mode
        model.train()

        # Append accuracy values to lists
        train_accuracy_list.append(train_accuracy)
        validation_accuracy_list.append(validation_accuracy)

    # Calculate and append the average loss for the epoch
    train_loss_list.append(sum(per_epoch_loss_list) / len(per_epoch_loss_list))

In [ ]:
# Plot training and validation accuracy over epochs
plt.plot([i for i in range(len(train_accuracy_list))], train_accuracy_list, label="Train")
plt.plot([i for i in range(len(validation_accuracy_list))], validation_accuracy_list, label="Validation")
plt.legend(loc="upper left")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.show()

**Dropout In Neural Networks**

In [ ]:
number_of_input_features = 27  # Number of input features in the dataset

# Reduced number of hidden units to 5 and just using 1 hidden layer
number_of_hidden_units = 5  # Number of hidden units in the neural network
epochs = 1000  # Number of training epochs
learning_rate = 0.01  # Learning rate for the optimizer
batch_size = 32  # Number of samples in each training batch
dropout_probablity = 0.6  # Probability of dropping out a neuron in dropout layer

# Convert training and testing data to PyTorch tensors
X_train_tensor = t.Tensor(X_train).type(t.float32)
Y_train_tensor = t.Tensor(Y_train).type(t.float32)
X_test_tensor = t.Tensor(X_test).type(t.float32)
Y_test_tensor = t.Tensor(Y_test).type(t.float32)

# Create a training dataset and data loader
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
train_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Define the neural network model
model = Sequential(
    Linear(number_of_input_features, number_of_hidden_units),  # Linear layer for matrix multiplication and bias addition
    Tanh(),  # Add Tanh activation function
    Dropout(dropout_probablity),  # Dropout layer to prevent overfitting
    Linear(number_of_hidden_units, 1),  # Another linear layer
    Sigmoid()  # Sigmoid activation for probability output
)

# Define the optimizer
optimizer = Adam(model.parameters(), lr=learning_rate)

# Define the loss function as Binary Cross-Entropy Loss
loss_fn = t.nn.BCELoss()

train_accuracy_list = []  # List to store training accuracy
validation_accuracy_list = []  # List to store validation accuracy

# Training loop
for epoch in range(epochs):
    if epoch % 500 == 0:
        learning_rate *= 0.9  # Learning rate scheduling

    per_epoch_loss_list = []  # List to store losses for each epoch

    for batch_idx, (X, Y) in enumerate(train_data_loader):
        # Forward pass: Compute predictions
        probs = model(X)

        # Calculate the loss
        loss = loss_fn(probs.view(-1), Y)
        per_epoch_loss_list.append(loss.item())

        # Backward pass: Compute gradients and update weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate the model on training and validation data
    with t.no_grad():
        # Set the model in eval mode; some layers use this for certain calculations during training
        model.eval()

        # Calculate accuracy on train data
        probs = model(X_train_tensor)
        prediction = (probs >= 0.5).type(t.LongTensor).view(-1)
        train_accuracy = (prediction == Y_train_tensor).type(t.float32).mean().item()

        # Calculate accuracy on validation data
        probs = model(X_test_tensor)
        prediction = (probs >= 0.5).type(t.LongTensor).view(-1)
        validation_accuracy = (prediction == Y_test_tensor).type(t.float32).mean().item()

        print(f'epoch {epoch}/{epochs} ---> train_accuracy: {train_accuracy}, validation_accuracy: {validation_accuracy}')

        # Set the model back to train mode
        model.train()

        # Append accuracy values to lists
        train_accuracy_list.append(train_accuracy)
        validation_accuracy_list.append(validation_accuracy)


In [ ]:
# Plot training and validation accuracy over epochs
plt.plot([i for i in range(len(train_accuracy_list))], train_accuracy_list, label="train")
plt.plot([i for i in range(len(validation_accuracy_list))], validation_accuracy_list, label="validation")
plt.legend(loc="upper left")
plt.xlabel('Epochs')
plt.ylabel('accuracy')

**Halting the Overfit: Early Stopping in Neural Networks**

In [ ]:
import torch as t
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import Sequential, Linear, Tanh, Sigmoid
from torch.optim import Adam
from torch.nn import BCELoss
import matplotlib.pyplot as plt

# Defining the parameters
number_of_input_features = 27
number_of_hidden_units = 27
epochs = 1000
learning_rate = 0.01
batch_size = 32
patience = 10 # adding a new hyperparameter for early stopping

# Loading and preprocessing the data
X_train_tensor = t.Tensor(X_train).type(t.float32)
Y_train_tensor = t.Tensor(Y_train).type(t.float32)

X_test_tensor = t.Tensor(X_test).type(t.float32)
Y_test_tensor = t.Tensor(Y_test).type(t.float32)

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
train_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# setting things up for early stopping
best_val_loss = float('inf')
trigger_times = 0

# Creating a neural network model
model = Sequential(
    Linear(number_of_input_features, number_of_hidden_units),
    Tanh(),
    Linear(number_of_hidden_units, 1),
    Sigmoid()
)

# Setting up the optimizer and loss function
optimizer = Adam(model.parameters(), lr=learning_rate)
loss_fn = BCELoss()

# Lists to store accuracy values during training
train_accuracy_list = []
validation_accuracy_list = []

# Main training loop
for epoch in range(epochs):
    # Learning rate scheduling (optional)
    if epoch % 500 == 0:
        learning_rate *= .9

    per_epoch_loss_list = []

    for batch_idx, (X, Y) in enumerate(train_data_loader):
        probs = model(X)

        # new loss is the old loss + regularization term
        loss = loss_fn(probs.view(-1), Y)

        per_epoch_loss_list.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    with t.no_grad():
        # Calculate accuracy on train data
        model.eval()
        probs = model(X_train_tensor)
        prediction = (probs >= .5).type(t.LongTensor).view(-1)

        train_accuracy = (prediction == Y_train_tensor).type(t.float32).mean().item()

        # Calculate accuracy on validation data
        probs = model(X_test_tensor)
        prediction = (probs > .5).type(t.LongTensor).view(-1)
        val_loss = loss_fn(probs.view(-1), Y_test_tensor).item() # validation loss

        validation_accuracy = (prediction == Y_test_tensor).type(t.float32).mean().item()

        print(f'epoch {epoch}/{epochs} ---> train_accuracy : {train_accuracy} , validation_accuracy : {validation_accuracy}')
        model.train()
        train_accuracy_list.append(train_accuracy)
        validation_accuracy_list.append(validation_accuracy)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            trigger_times = 0  # Reset trigger counter if validation loss improved
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f'Early stopping at epoch {epoch} due to no improvement in validation loss.')
                break

In [ ]:
# Plotting the training and validation accuracy
plt.plot([i for i in range(len(train_accuracy_list))], train_accuracy_list, label="train")
plt.plot([i for i in range(len(validation_accuracy_list))], validation_accuracy_list, label='validation')
plt.legend(loc="upper left")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')

